In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate
!pip install autoawq

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-943vzuyu
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-943vzuyu
  Resolved https://github.com/huggingface/transformers to commit b71de73468429eb02da18caa50e9b5200400a4ed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11553909 sha256=1a31002611289dd16aaef64833fa17fb77f8da3f265f1dabc279ec84099f6af0
  Stored in directory: /tmp/pip-ephem-wheel-cache-hdf8_lcu/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-8B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import torch

def inference_text_with_budget(model, tokenizer, prompt, thinking_budget, max_new_tokens):

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_length = model_inputs.input_ids.size(-1)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=thinking_budget,
        do_sample = False
    )
    output_ids = generated_ids[0][input_length:].tolist()

    # check if the generation has already finished (151645 is <|im_end|>)
    if 151645 not in output_ids:
        # check if the thinking process has finished (151668 is </think>)
        # and prepare the second model input
        if 151668 not in output_ids:
            print("thinking budget is reached")
            '''early_stopping_text = (
            "\n</think>\n"
            "FINAL ANSWER PROTOCOL ENGAGED.\n"
            "The conversion phase is complete. Output the final SPICE netlist based on the provided text description and the preceding analysis.\n"
            "Do not add further commentary. The output must be the SPICE netlist ONLY.\n"
            )'''
            early_stopping_text = (
    "\n</think>\n"
    "FINAL ANSWER PROTOCOL ENGAGED.\n"
    "The reasoning phase has been forcibly stopped due to token budget limits.\n"
    "Before producing the final output, perform an internal check for repetition or looping patterns "
    "in the preceding reasoning (e.g., repeated phrases, redundant chains, or self-referential reasoning). "
    "Ignore and discard any such repetitive or looping segments.\n"
    "Now, synthesize the final SPICE netlist using only the coherent and non-repetitive analytical content "
    "from the reasoning above.\n"
    "Do not include commentary or explanations. The output must be the SPICE netlist ONLY.\n"
)
            early_stopping_ids = tokenizer([early_stopping_text], return_tensors="pt", return_attention_mask=False).input_ids.to(model.device)
            input_ids = torch.cat([generated_ids, early_stopping_ids], dim=-1)
        else:
            input_ids = generated_ids
        attention_mask = torch.ones_like(input_ids, dtype=torch.int64)

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=input_length + max_new_tokens - input_ids.size(-1),
            do_sample = False
        )
        output_ids = generated_ids[0][input_length:].tolist()

    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print(f"thinking content: {thinking_content}")
    print('-------------------')

    return content


In [ ]:
import os

files_directory = "/content/drive/MyDrive/GLM_description"
output_directory = "results/" # Directory where the text files are saved

# Make sure the output directory exists
os.makedirs(output_directory, exist_ok=True)



for filename in os.listdir(files_directory):
    if filename.lower().endswith('.txt'):
        base_filename = os.path.splitext(filename)[0]
        print(f"Processing file: {filename}")
        file_path = os.path.join(files_directory, filename)
        print(file_path)
        with open(file_path, 'r', encoding='utf-8') as f:
            description = f.read()




        prompt = f'''
                You are an expert electronics engineer specializing in converting natural language circuit descriptions into precise, topology-focused SPICE netlists. Your task is to analyze the provided circuit description and generate a clean SPICE netlist by following a strict internal protocol.

        **OUTPUT FORMAT:**
        - The output MUST BE ONLY the SPICE netlist.
        - Do NOT include any additional text, explanations, greetings, or markdown code blocks like ```spice ... ```.

        ---
        **STRICT INTERNAL ANALYSIS PROTOCOL**
        You MUST follow this internal thought process before and during generation.

        **PHASE 1: Silent Net Inventory**
        - Before writing any output, first silently read the entire description and identify all unique net names mentioned (e.g., Vin, Vout, VDD, ground). This is your internal reference map for ensuring consistency.
        - Use '0' for any net described as ground, GND, or VSS.

        **PHASE 2: Component-by-Component Netlist Generation**
        - After your silent net inventory, process each component mentioned in the description one by one to generate its corresponding SPICE netlist line.

        **Analysis Sub-Protocol for MOSFETs (CRITICAL)**
        - Before writing the SPICE line for ANY MOSFET, you MUST perform this internal analysis:
            1.  **Read and Extract:** Carefully re-read the part of the description that specifies the MOSFET's connections.
            2.  **Map Pins to Nets:** Mentally create a precise mapping for each terminal based on what you read. For example: `Gate -> Vin`, `Source -> 0`, `Drain -> Vout`.
            3.  **Generate the SPICE Line:** Only after this mapping is clear in your mind, write the SPICE line using the mapped nets in the STRICT order: `M# <Drain_Net> <Gate_Net> <Source_Net> <Bulk_Net> MODEL_TYPE`.

        ---
        **GENERAL RULES TO APPLY:**
        1.  **Component Naming:** Use the component names mentioned in the description (e.g., M1, R1).
        2.  **Component Values:** Extract values (e.g., '1k') if clearly mentioned. If not specified, use 'VALUE'.
        3.  **MOSFET Bulk Connection:** If the description does not specify a bulk (or body) connection, you MUST connect the Bulk (B) terminal to the SAME net as the Source (S).
        4.  **EXCLUSION RULES:** Do not generate `.MODEL`, `.OP`, `.END`, or `.SUBCKT` lines. Only generate the component definition lines.

        ---
        **EXAMPLE OF CORRECT EXECUTION:**
        *   **If the input description is:**
            "This circuit features an NMOS transistor, M1, whose gate is tied to the 'Vin' node. Its source is connected to ground. The drain of M1 is connected to the 'Vout' node. A pull-up resistor, R1, with a value of 1k ohm, is placed between the VDD supply net and the 'Vout' node."
        *   **The ONLY correct output is:**
        M1 Vout Vin 0 0 NMOS
        R1 VDD Vout 1k

        ---
        Now, analyze the following circuit description and generate the SPICE netlist by STRICTLY following the protocol above.

        **CIRCUIT DESCRIPTION:**
        {description}
                '''

        netlist = inference_text_with_budget(model, tokenizer, prompt, 6000, 8192)

        print(netlist)

        output_filename = f"{base_filename}_netlist_from_descr.txt"

        output_path = os.path.join(output_directory, output_filename)


        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(netlist)

        '''output_filename = f"{base_filename}_CoT.txt"

        output_path = os.path.join(output_directory, output_filename)

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(CoT)


        print(f"Response saved to: {output_path}\n")'''

Processing file: 768.txt
/content/drive/MyDrive/GLM_description/768.txt


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


thinking budget is reached
thinking content: <think>
Okay, let's tackle this SPICE netlist generation. First, I need to follow the strict protocol outlined. Let me start by going through the provided analysis step by step.

**Phase 1: Net Inventory**  
The primary nets are VDD and 0 (ground). There are two unlabeled nodes, n1 and n2, which connect components. So the net list is VDD, 0, n1, n2. Got that.

**Phase 2: Component Analysis**  
Starting with M1, a PMOS. The description says the gate is connected to n2, source and bulk to 0, and drain to n2. Wait, the drain and gate are both connected to n2? That seems odd. But according to the analysis, that's how it's mapped. So the SPICE line for M1 would be M1 Drain Gate Source Bulk. Wait, the order is Drain, Gate, Source, Bulk. Wait, the general rule for MOSFETs is Drain, Gate, Source, Bulk. But the user's example had M1 Vout Vin 0 0 NMOS. So the order is Drain, Gate, Source, Bulk. So for M1, Drain is n2, Gate is n2, Source is 0, Bulk is 